# 0. Project outline

**Problem:** RAM has a limited number of frames, so a requested page may need to replace a page that is already in memory.

**Objectives:** Learn how FIFO, LRU, Optimal, and Clock make replacement choices; record hits, faults, and evictions; compare the results.

**What we implement now:** A small simulation of page replacement logic with a step table and charts. This is **not a measurement of a real computer**.

**Planned for later:** Explore memory scaling, Belady's anomaly, workloads, dirty-page write-back, and thrashing.

**Text architecture:** `reference string -> algorithm -> steps -> table / charts`

## Key terms

- **Page:** A fixed-size unit of a program's memory.
- **Frame:** A slot in RAM that can hold one page.
- **Page hit:** The requested page is already in a frame.
- **Page fault:** The requested page is not in a frame and must be loaded.
- **Page replacement:** Removing a resident page to make room for a requested page.
- **Reference string:** The ordered list of pages requested by a program.
- **Hit ratio:** Hits divided by the total number of requests.
- **Fault ratio:** Faults divided by the total number of requests.
- **Locality:** The tendency for programs to reuse nearby pages or pages used recently.

# 1. Setup

Import the three allowed libraries and define a small parser. The two inputs below are used by the visualizer and comparison later; edit them and re-run the notebook from the visualizer cell onward.

In [ ]:
import numpy as np  # Use NumPy for the frame-grid chart.
import pandas as pd  # Use pandas to display step and comparison tables.
import matplotlib.pyplot as plt  # Use Matplotlib to draw charts.


def parse_reference_string(text):
    """Convert space-separated page numbers into a list of integers."""
    # Split on whitespace so one or more spaces are accepted.
    parts = text.split()
    # Reject an empty request sequence.
    if not parts:
        raise ValueError("Enter at least one page number.")
    # Convert every page number and report invalid text clearly.
    try:
        refs = [int(part) for part in parts]
    except ValueError as error:
        raise ValueError("Use whole-number page values separated by spaces.") from error
    # Return the parsed reference string.
    return refs

In [ ]:
reference_string = "7 0 1 2 0 3 0 4 2 3 0 3 2 1 2 0 1 7 0 1"  # Edit this workload to experiment.
number_of_frames = 3  # Edit this RAM capacity to experiment.

# 2. The four algorithms

Each algorithm returns one dictionary for every request. `frames` is a snapshot of RAM after that request.

## FIFO

**Rule:** Replace the page that has been in RAM the longest.

**By hand:** With two frames, requests `1, 2, 3` load 1, then 2, then replace 1 with 3.

**Strength:** Very simple to understand and implement. **Weakness:** It ignores how recently or often a page was used.

In [ ]:
def fifo(refs, n_frames):
    """Simulate First-In, First-Out page replacement."""
    # Require at least one RAM frame.
    if n_frames < 1:
        raise ValueError("n_frames must be at least 1.")
    frames = [None] * n_frames  # None marks an empty frame.
    arrival_order = []  # Store resident pages from oldest to newest.
    steps = []  # Collect one result record for every request.
    for page in refs:
        hit = page in frames  # Check whether the page is already resident.
        evicted = None  # A hit or empty-frame load evicts nothing.
        if hit:
            reason = f"Page {page} is already in RAM, so this request is a hit."
        else:
            empty_index = frames.index(None) if None in frames else None  # Find an unused slot.
            if empty_index is not None:
                frames[empty_index] = page  # Fill an empty frame first.
                arrival_order.append(page)  # Record when this page arrived.
                reason = f"Page {page} was not in RAM, so it was loaded into an empty frame."
            else:
                evicted = arrival_order.pop(0)  # The first arrival is the FIFO victim.
                frame_index = frames.index(evicted)  # Find the victim's frame.
                frames[frame_index] = page  # Load the requested page in its place.
                arrival_order.append(page)  # The new page joins the back of the queue.
                reason = f"FIFO evicts {evicted} because it arrived first."
        steps.append({"page": page, "frames": frames.copy(), "hit": hit, "evicted": evicted, "reason": reason})  # Save a snapshot.
    return steps  # Give the complete simulation back to the caller.

## LRU

**Rule:** Replace the page whose most recent use was the longest time ago.

**By hand:** With two frames, requests `1, 2, 1, 3` make 1 recent, so request 3 replaces 2.

**Strength:** Uses recent behavior and often works well for programs with locality. **Weakness:** It must remember recent use and cannot know every workload perfectly.

In [ ]:
def lru(refs, n_frames):
    """Simulate Least Recently Used page replacement."""
    # Require at least one RAM frame.
    if n_frames < 1:
        raise ValueError("n_frames must be at least 1.")
    frames = [None] * n_frames  # None marks an empty frame.
    last_used = {}  # Map each resident page to its most recent request index.
    steps = []  # Collect one result record for every request.
    for index, page in enumerate(refs):
        hit = page in frames  # Check whether the page is already resident.
        evicted = None  # A hit or empty-frame load evicts nothing.
        if hit:
            reason = f"Page {page} is already in RAM, so this request is a hit."
        else:
            empty_index = frames.index(None) if None in frames else None  # Find an unused slot.
            if empty_index is not None:
                frames[empty_index] = page  # Fill an empty frame first.
                reason = f"Page {page} was not in RAM, so it was loaded into an empty frame."
            else:
                evicted = min(frames, key=lambda resident: last_used[resident])  # Pick the oldest last-use index.
                steps_ago = index - last_used[evicted]  # Measure the time since that page was used.
                frame_index = frames.index(evicted)  # Find the least-recent page's frame.
                frames[frame_index] = page  # Load the requested page in its place.
                reason = f"LRU evicts {evicted} because it was last used {steps_ago} steps ago."
        last_used[page] = index  # A request makes this page the most recently used.
        steps.append({"page": page, "frames": frames.copy(), "hit": hit, "evicted": evicted, "reason": reason})  # Save a snapshot.
    return steps  # Give the complete simulation back to the caller.

## Optimal

**Rule:** Replace the resident page whose next use is farthest in the future, or that is never used again.

**By hand:** With two frames, requests `1, 2, 1, 3, 2` make request 3 replace 1, because 2 is needed sooner.

**Strength:** Gives the fewest possible faults for a known sequence. **Weakness:** A real operating system cannot see future requests, so it is a benchmark, not a practical policy.

In [ ]:
def optimal(refs, n_frames):
    """Simulate Optimal page replacement using the known future requests."""
    # Require at least one RAM frame.
    if n_frames < 1:
        raise ValueError("n_frames must be at least 1.")
    frames = [None] * n_frames  # None marks an empty frame.
    steps = []  # Collect one result record for every request.
    for index, page in enumerate(refs):
        hit = page in frames  # Check whether the page is already resident.
        evicted = None  # A hit or empty-frame load evicts nothing.
        if hit:
            reason = f"Page {page} is already in RAM, so this request is a hit."
        else:
            empty_index = frames.index(None) if None in frames else None  # Find an unused slot.
            if empty_index is not None:
                frames[empty_index] = page  # Fill an empty frame first.
                reason = f"Page {page} was not in RAM, so it was loaded into an empty frame."
            else:
                future = refs[index + 1:]  # Look only at requests after this one.
                next_use = {resident: (future.index(resident) if resident in future else None) for resident in frames}  # Find each page's next use.
                evicted = max(frames, key=lambda resident: float("inf") if next_use[resident] is None else next_use[resident])  # Choose the farthest or never-used page.
                frame_index = frames.index(evicted)  # Find the chosen victim's frame.
                frames[frame_index] = page  # Load the requested page in its place.
                if next_use[evicted] is None:
                    reason = f"Optimal evicts {evicted} because it is not needed again."
                else:
                    steps_ahead = next_use[evicted] + 1  # Count requests until the page is needed again.
                    reason = f"Optimal evicts {evicted} because it is not needed again for {steps_ahead} steps."
        steps.append({"page": page, "frames": frames.copy(), "hit": hit, "evicted": evicted, "reason": reason})  # Save a snapshot.
    return steps  # Give the complete simulation back to the caller.

## Clock

**Rule:** A moving pointer looks for a page with reference bit 0, clearing bit 1 values as it passes them.

**By hand:** With two frames containing pages 1 and 2 and both bits 1, a replacement clears both bits on its scan and evicts the first page it reaches again with bit 0.

**Strength:** Uses little information and approximates LRU efficiently. **Weakness:** It only approximates recency, so its decisions may differ from true LRU.

In [ ]:
def clock(refs, n_frames):
    """Simulate Clock replacement with one reference bit per frame."""
    # Require at least one RAM frame.
    if n_frames < 1:
        raise ValueError("n_frames must be at least 1.")
    frames = [None] * n_frames  # None marks an empty frame.
    reference_bits = [0] * n_frames  # A bit of 1 means recently referenced.
    pointer = 0  # The pointer marks where the next Clock scan begins.
    steps = []  # Collect one result record for every request.
    for page in refs:
        hit = page in frames  # Check whether the page is already resident.
        evicted = None  # A hit or empty-frame load evicts nothing.
        if hit:
            frame_index = frames.index(page)  # Find the requested page's frame.
            reference_bits[frame_index] = 1  # Give a hit a second chance.
            reason = f"Page {page} is already in RAM, so this request is a hit."
        else:
            empty_index = frames.index(None) if None in frames else None  # Find an unused slot.
            if empty_index is not None:
                frames[empty_index] = page  # Fill an empty frame first.
                reference_bits[empty_index] = 1  # Mark the newly loaded page as referenced.
                reason = f"Page {page} was not in RAM, so it was loaded into an empty frame."
            else:
                cleared_pages = []  # Remember which reference bits were cleared during this scan.
                while reference_bits[pointer] == 1:
                    cleared_pages.append(frames[pointer])  # Record the page receiving a second chance.
                    reference_bits[pointer] = 0  # Clear its reference bit.
                    pointer = (pointer + 1) % n_frames  # Move to the next frame.
                evicted = frames[pointer]  # The first zero-bit page is the victim.
                frame_index = pointer  # Keep the victim's slot for the new page.
                frames[frame_index] = page  # Load the requested page.
                reference_bits[frame_index] = 1  # Give the new page a reference bit of 1.
                pointer = (pointer + 1) % n_frames  # Continue the next scan after this slot.
                if cleared_pages:
                    cleared_text = ", ".join(str(value) for value in cleared_pages)  # Make cleared pages readable.
                    reason = f"Clock cleared the reference bit of {cleared_text}, then evicted {evicted} whose bit was 0."
                else:
                    reason = f"Clock evicted {evicted} whose reference bit was 0."
        steps.append({"page": page, "frames": frames.copy(), "hit": hit, "evicted": evicted, "reason": reason})  # Save a snapshot.
    return steps  # Give the complete simulation back to the caller.

# 3. Validation

Check the required example. A fault is any request that was not already in a frame.

In [ ]:
validation_refs = parse_reference_string("7 0 1 2 0 3 0 4 2 3 0 3 2 1 2 0 1 7 0 1")  # Use the specified test string.
fifo_validation = fifo(validation_refs, 3)  # Run FIFO with three frames.
lru_validation = lru(validation_refs, 3)  # Run LRU with three frames.
optimal_validation = optimal(validation_refs, 3)  # Run Optimal with three frames.
clock_validation = clock(validation_refs, 3)  # Run Clock with three frames.
assert sum(not step["hit"] for step in fifo_validation) == 15  # FIFO must have 15 faults.
assert sum(not step["hit"] for step in lru_validation) == 12  # LRU must have 12 faults.
assert sum(not step["hit"] for step in optimal_validation) == 9  # Optimal must have 9 faults.
print("FIFO faults:", sum(not step["hit"] for step in fifo_validation))  # Show the checked FIFO count.
print("LRU faults:", sum(not step["hit"] for step in lru_validation))  # Show the checked LRU count.
print("Optimal faults:", sum(not step["hit"] for step in optimal_validation))  # Show the checked Optimal count.
print("Clock faults:", sum(not step["hit"] for step in clock_validation))  # Also show Clock's result.
print("All required asserts passed.")  # Confirm the validation completed.

# 4. Step-by-step visualizer

The selected algorithm produces a colored table and a frame grid. Red means fault; green means hit. The request and frame settings come from the editable inputs near the top.

In [ ]:
refs = parse_reference_string(reference_string)  # Read the current editable workload.
visualizer_algorithm = lru  # Choose fifo, lru, optimal, or clock for the step display.
visualizer_name = "LRU"  # Use the matching name in the chart title.
visualizer_steps = visualizer_algorithm(refs, number_of_frames)  # Run the selected policy.
visualizer_rows = []  # Build plain rows before making a DataFrame.
for step_number, step in enumerate(visualizer_steps, start=1):
    visualizer_rows.append({
        "step": step_number,
        "request": step["page"],
        "RAM frames": str(step["frames"]),
        "result": "Hit" if step["hit"] else "Fault",
        "evicted": "-" if step["evicted"] is None else step["evicted"],
        "reason": step["reason"],
    })  # Keep the table columns easy to explain.
visualizer_table = pd.DataFrame(visualizer_rows)  # Make the step table.
def color_result(row):
    """Color each table row green for a hit or red for a fault."""
    # Use a light fill so the text remains readable.
    color = "#d9ead3" if row["result"] == "Hit" else "#f4cccc"
    # Return one style value for every cell in the row.
    return [f"background-color: {color}"] * len(row)
display(visualizer_table.style.apply(color_result, axis=1))  # Show the colored table.

frame_grid = np.array([step["frames"] for step in visualizer_steps], dtype=object).T  # Arrange frames by row and request by column.
grid_colors = np.zeros((number_of_frames, len(visualizer_steps), 3))  # Start every grid cell with a neutral color.
for column, step in enumerate(visualizer_steps):
    color = [0.90, 0.45, 0.45] if not step["hit"] else [0.45, 0.75, 0.50]  # Choose fault red or hit green.
    grid_colors[:, column, :] = color  # Color all frames for this request.
fig, ax = plt.subplots(figsize=(max(8, len(visualizer_steps) * 0.55), max(2, number_of_frames * 0.65)))  # Size the chart for the workload.
ax.imshow(grid_colors, aspect="auto")  # Draw one colored cell per frame and request.
ax.set_xticks(range(len(visualizer_steps)))  # Place a tick for every request.
ax.set_xticklabels([str(step["page"]) for step in visualizer_steps])  # Label columns with requested pages.
ax.set_yticks(range(number_of_frames))  # Place a tick for every RAM frame.
ax.set_yticklabels([f"Frame {index + 1}" for index in range(number_of_frames)])  # Label the RAM rows.
ax.set_xlabel("Request (page number)")  # Explain the horizontal axis.
ax.set_title(f"{visualizer_name}: RAM after each request (red=fault, green=hit)")  # Explain the chart colors.
ax.set_xticks(np.arange(-0.5, len(visualizer_steps), 1), minor=True)  # Add vertical cell borders.
ax.set_yticks(np.arange(-0.5, number_of_frames, 1), minor=True)  # Add horizontal cell borders.
ax.grid(which="minor", color="white", linewidth=2)  # Separate the frame cells.
ax.tick_params(which="minor", bottom=False, left=False)  # Hide extra border ticks.
for column, step in enumerate(visualizer_steps):
    for row, page in enumerate(step["frames"]):
        ax.text(column, row, "-" if page is None else str(page), ha="center", va="center")  # Print each resident page.
plt.tight_layout()  # Prevent labels from overlapping.
plt.show()  # Display the frame grid.
plt.close(fig)  # Release this figure before drawing another chart.

# 5. Algorithm comparison

Compare all four policies using the same reference string and frame count. Optimal is a benchmark only because it needs future knowledge. The best practical choice is picked from FIFO, LRU, and Clock by the fewest faults in this workload.

In [ ]:
comparison_refs = parse_reference_string(reference_string)  # Read the current editable workload.
algorithms = {"FIFO": fifo, "LRU": lru, "Optimal": optimal, "Clock": clock}  # Pair each label with its function.
comparison_rows = []  # Collect one summary row per policy.
for name, algorithm in algorithms.items():
    algorithm_steps = algorithm(comparison_refs, number_of_frames)  # Run every policy on the shared workload.
    faults = sum(not step["hit"] for step in algorithm_steps)  # Count requests that were not resident.
    hits = sum(step["hit"] for step in algorithm_steps)  # Count requests that were resident.
    total = len(algorithm_steps)  # Count every request once.
    comparison_rows.append({
        "Algorithm": name,
        "Faults": faults,
        "Hits": hits,
        "Hit ratio": hits / total,
        "Fault ratio": faults / total,
    })  # Save this policy's summary values.
comparison_table = pd.DataFrame(comparison_rows).set_index("Algorithm")  # Make the comparison table.
display(comparison_table.style.format({"Hit ratio": "{:.1%}", "Fault ratio": "{:.1%}"}))  # Show ratios as percentages.
comparison_table["Faults"].plot(kind="bar", color=["#5b9bd5", "#70ad47", "#a5a5a5", "#ed7d31"])  # Compare fault counts visually.
plt.ylabel("Page faults")  # Name the vertical axis.
plt.title("Page faults by replacement algorithm")  # Add a descriptive title.
plt.xticks(rotation=0)  # Keep policy labels horizontal.
plt.tight_layout()  # Keep chart labels inside the figure.
plt.show()  # Display the bar chart.
plt.close()  # Release the bar-chart figure.
practical_results = comparison_table.loc[["FIFO", "LRU", "Clock"], "Faults"]  # Exclude the future-aware benchmark.
best_fault_count = practical_results.min()  # Find the lowest practical fault count.
best_choices = practical_results[practical_results == best_fault_count].index.tolist()  # Keep ties if policies match.
print("Best practical choice:", " / ".join(best_choices), f"({best_fault_count} faults on this workload)")  # State the workload-specific result.

# 6. Try it yourself

Edit the two variables in the input cell near the top: `reference_string` and `number_of_frames`. Re-run that input cell, then re-run the visualizer and comparison cells in sections 4 and 5. The required validation cell always uses its own fixed example.

# 7. Summary and viva questions

1. **Why is Optimal not realistic?** It needs to know future page requests, which a running program has not made yet.
2. **How does Clock approximate LRU?** It gives recently referenced pages a second chance and scans past them by clearing their bits.
3. **What is a page fault?** A requested page is absent from RAM and must be loaded.
4. **Why do more frames usually help?** More frames can keep more pages resident and may prevent replacement faults.
5. **What does FIFO use to choose a victim?** It replaces the page that arrived earliest and is still in RAM.
6. **What is locality?** Programs often reuse pages they used recently or pages close to the current work.

# 8. Future extensions

- **Memory scaling:** Show how fault counts change as the number of frames grows.
- **Belady's anomaly:** Demonstrate that FIFO can sometimes have more faults after adding a frame.
- **Workload analysis:** Compare different reference strings and their locality patterns.
- **Dirty-page write-back:** Show the extra work required when an evicted page has been modified.
- **Thrashing:** Illustrate excessive faults when the working set does not fit in available frames.